__Paper__: Using neural networks as an alternative to air dispersion modeling in environmental impact assessment.

__Authors__: Mateo Concha and Gonzalo A. Ruz.

__Description__: Code used for the paper development.

In [ ]:
pip install tqdm

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# --- Load datasets (adjust folder if needed) ---
train_data     = pd.read_csv("train.csv")
validate_data  = pd.read_csv("validation.csv")
test_FFEE_data = pd.read_csv("test_FFEE.csv")
test_CV_data   = pd.read_csv("test_CV.csv")

# --- Explicit target columns
target_ffee = "FFEE"
target_cv   = "Chacaya"

# --- Sanity checks (gives a clear error message if files differ) ---
for name, df, target in [
    ("train", train_data, target_ffee),
    ("validation", validate_data, target_ffee),
    ("test_FFEE", test_FFEE_data, target_ffee),
    ("test_CV", test_CV_data, target_cv),
]:
    if target not in df.columns:
        raise KeyError(
            f"{name}: expected target column '{target}' not found.\n"
            f"Available columns: {list(df.columns)}"
        )

# --- Split into X/y ---
X_train, y_train = train_data.drop(columns=[target_ffee]), train_data[target_ffee]
X_val,   y_val   = validate_data.drop(columns=[target_ffee]), validate_data[target_ffee]

X_test_FFEE, y_test_FFEE = test_FFEE_data.drop(columns=[target_ffee]), test_FFEE_data[target_ffee]
X_test_CV,   y_test_CV   = test_CV_data.drop(columns=[target_cv]),   test_CV_data[target_cv]

print("Shapes:",
      "train", X_train.shape,
      "val", X_val.shape,
      "test_FFEE", X_test_FFEE.shape,
      "test_CV", X_test_CV.shape)

# --- Scaling (fit on TRAIN only; apply to all) ---
scaler = MinMaxScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test_FFEE = scaler.transform(X_test_FFEE)
X_test_CV   = scaler.transform(X_test_CV)


In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

RANDOM_STATE = 123

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def report(name, y_true, y_pred):
    print(f"{name}: RMSE={rmse(y_true, y_pred):.3f} | MAE={mean_absolute_error(y_true, y_pred):.3f} | R^2={r2_score(y_true, y_pred):.3f}")


pipe = Pipeline([
    ("scaler", StandardScaler()),  
    ("rf", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

param_distributions = {
    "rf__n_estimators": randint(50, 201),
    "rf__max_depth": randint(5, 41),         
    "rf__max_features": randint(3, X_train.shape[1] + 1),
    "rf__min_samples_split": randint(2, 11),
    "rf__min_samples_leaf": randint(1, 6),
}

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_distributions,
    n_iter=60,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)
print("Best CV params:", search.best_params_)
print("Best CV RMSE:", -search.best_score_)

# Evaluate chosen model on the fixed validation.csv
best_model = search.best_estimator_
val_pred = best_model.predict(X_val)
report("Validation (fixed split)", y_val, val_pred)


In [ ]:
# Refit on train+val (for final baseline numbers)
X_tr_all = np.vstack([X_train, X_val])
y_tr_all = np.concatenate([y_train, y_val])

final_model = search.best_estimator_
final_model.fit(X_tr_all, y_tr_all)

pred_test_f = final_model.predict(X_test_FFEE)
pred_test_cv = final_model.predict(X_test_CV)

report("Test FFEE", y_test_FFEE, pred_test_f)
report("Test CV", y_test_CV, pred_test_cv)
